In [2]:
import struct
import socket

In [4]:
def build_dns_query(domain_name, query_type=1):
    """
    Constrói uma mensagem de consulta DNS (A record) conforme RFC 1035.
    """
    # 1. Cabeçalho (Header)
    # ID: Random, Flags: 0x0100 (Recursion Desired), QDCOUNT: 1, ANCOUNT: 0, NSCOUNT: 0, ARCOUNT: 0
    transaction_id = 0xAAAA
    flags = 0x0100
    qdcount = 1
    ancount = 0
    nscount = 0
    arcount = 0
    
    # '!' = network order (big-endian), H = unsigned short (2 bytes)
    header = struct.pack('!HHHHHH', transaction_id, flags, qdcount, ancount, nscount, arcount)

    # 2. Pergunta (Question) - Nome codificado (ex: www.google.com -> \x03www\x06google\x03com\x00)
    encoded_name = b""
    for part in domain_name.encode("ascii").split(b"."):
        encoded_name += bytes([len(part)]) + part
    encoded_name += b"\x00"  # Finalizador

    # Tipo: A (1) | Classe: IN (1)
    qtype = query_type
    qclass = 1 # IN
    
    question = encoded_name + struct.pack('!HH', qtype, qclass)

    return header + question

domain = "google.com"
dns_message = build_dns_query(domain)

print(f"Mensagem DNS para {domain} ({len(dns_message)} bytes):")
print(dns_message)

Mensagem DNS para google.com (28 bytes):
b'\xaa\xaa\x01\x00\x00\x01\x00\x00\x00\x00\x00\x00\x06google\x03com\x00\x00\x01\x00\x01'


In [ ]:
dns_query = dns_message

DNS_SERVER = "8.8.8.8"
PORT = 53

# 3. Criar socket UDP (SOCK_DGRAM)
# Note: Raw sockets (SOCK_RAW) costumam exigir privilégios de root/admin,
# mas SOCK_DGRAM envia o payload DNS "cru" dentro de um UDP gerado pelo SO.
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

try:
    print(f"Enviando query para {DNS_SERVER}...")
    
    # 4. Enviar a mensagem pronta
    sock.sendto(dns_query, (DNS_SERVER, PORT))
    
    # 5. Receber resposta
    data, addr = sock.recvfrom(1024)

finally:
    sock.close()


Enviando query para 8.8.8.8...
Resposta recebida de ('8.8.8.8', 53):
aaaa8180000100060000000006676f6f676c6503636f6d0000010001c00c000100010000012c0004c0b29b8bc00c000100010000012c0004c0b29b8ac00c000100010000012c0004c0b29b66c00c000100010000012c0004c0b29b64c00c000100010000012c0004c0b29b65c00c000100010000012c0004c0b29b71


In [ ]:
DNS_TYPE_NAMES = {
    1: "A",
    2: "NS",
    5: "CNAME",
    6: "SOA",
    12: "PTR",
    15: "MX",
    16: "TXT",
    28: "AAAA",
}

def decode_dns_name(packet, offset):
    labels = []
    jumped = False
    next_offset = offset

    while True:
        length = packet[offset]

        if length & 0xC0 == 0xC0:
            pointer = ((length & 0x3F) << 8) | packet[offset + 1]
            if not jumped:
                next_offset = offset + 2
            offset = pointer
            jumped = True
            continue

        if length == 0:
            offset += 1
            if not jumped:
                next_offset = offset
            break

        offset += 1
        labels.append(packet[offset : offset + length].decode("ascii"))
        offset += length

    return ".".join(labels), next_offset

def decode_dns_response(packet):
    transaction_id, flags, qdcount, ancount, nscount, arcount = struct.unpack_from('!HHHHHH', packet, 0)
    offset = 12

    qr = bool(flags & 0x8000)
    aa = bool(flags & 0x0400)
    tc = bool(flags & 0x0200)
    rd = bool(flags & 0x0100)
    ra = bool(flags & 0x0080)
    rcode = flags & 0x000F

    lines = [
        f"ID da transação: 0x{transaction_id:04x}",
        f"Flags: 0x{flags:04x}",
        f"  - resposta: {qr}",
        f"  - autoritária: {aa}",
        f"  - truncada: {tc}",
        f"  - recursion desired: {rd}",
        f"  - recursion available: {ra}",
        f"  - rcode: {rcode}",
        f"Perguntas: {qdcount}",
        f"Respostas: {ancount}",
        f"Autoritativas: {nscount}",
        f"Adicionais: {arcount}",
        "",
        "Pergunta:",
    ]

    for index in range(qdcount):
        name, offset = decode_dns_name(packet, offset)
        qtype, qclass = struct.unpack_from('!HH', packet, offset)
        offset += 4
        lines.append(f"  {index + 1}. nome={name}, tipo={DNS_TYPE_NAMES.get(qtype, qtype)}, classe={qclass}")

    lines.extend(["", "Respostas:"])

    for index in range(ancount):
        name, offset = decode_dns_name(packet, offset)
        rtype, rclass, ttl, rdlength = struct.unpack_from('!HHIH', packet, offset)
        offset += 10
        rdata = packet[offset : offset + rdlength]
        offset += rdlength

        if rtype == 1 and rdlength == 4:
            rdata_text = socket.inet_ntoa(rdata)
        else:
            rdata_text = rdata.hex()

        lines.append(f"  {index + 1}. nome={name}, tipo={DNS_TYPE_NAMES.get(rtype, rtype)}, ttl={ttl}s, dado={rdata_text}")

    return "\n".join(lines)

print(f"Resposta recebida de {addr}:")
print(decode_dns_response(data))

Resposta recebida de ('8.8.8.8', 53):
aaaa8180000100060000000006676f6f676c6503636f6d0000010001c00c000100010000012c0004c0b29b8bc00c000100010000012c0004c0b29b8ac00c000100010000012c0004c0b29b66c00c000100010000012c0004c0b29b64c00c000100010000012c0004c0b29b65c00c000100010000012c0004c0b29b71
b'\xaa\xaa\x81\x80\x00\x01\x00\x06\x00\x00\x00\x00\x06google\x03com\x00\x00\x01\x00\x01\xc0\x0c\x00\x01\x00\x01\x00\x00\x01,\x00\x04\xc0\xb2\x9b\x8b\xc0\x0c\x00\x01\x00\x01\x00\x00\x01,\x00\x04\xc0\xb2\x9b\x8a\xc0\x0c\x00\x01\x00\x01\x00\x00\x01,\x00\x04\xc0\xb2\x9bf\xc0\x0c\x00\x01\x00\x01\x00\x00\x01,\x00\x04\xc0\xb2\x9bd\xc0\x0c\x00\x01\x00\x01\x00\x00\x01,\x00\x04\xc0\xb2\x9be\xc0\x0c\x00\x01\x00\x01\x00\x00\x01,\x00\x04\xc0\xb2\x9bq'


����      googlecom   �    , �����    , �����    , ���f�    , ���d�    , ���e�    , ���q
